In [1]:
# 02_ranking_stability.ipynb
# Level 2 (Theorem 2): ranking is robust to quantization. Measures AUC loss
# and rank-reversal rate versus Delta, and shows the corollary that higher
# discrimination buys greater robustness (smaller AUC loss at fixed Delta).

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def quant(A, D): return D * np.round(A / D)
def fq(D): return b0 + quant(C, D).sum(1)

Deltas = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.4])
scales = {"baseline": 1.0, "stronger": 1.6}   # scaling contributions raises discrimination

def reversal_rate(fc, fqv, npair=200000, seed=0):
    r = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    i = r.choice(pos, npair); j = r.choice(neg, npair)
    concordant = np.sign(fc[i] - fc[j]) > 0
    now_not = np.sign(fqv[i] - fqv[j]) <= 0
    return float(np.mean(concordant & now_not))

rows = []
for name, s in scales.items():
    Cs = C * s; fs = b0 + Cs.sum(1)
    a0 = roc_auc_score(y, fs)
    for D in Deltas:
        fqs = b0 + (D * np.round(Cs / D)).sum(1)
        rows.append(dict(level=name, auc0=a0, Delta=D,
                         auc_loss=a0 - roc_auc_score(y, fqs),
                         reversal=reversal_rate(fs, fqs)))
tab = pd.DataFrame(rows)
tab.to_csv("../results/tables/02_ranking_stability.csv", index=False)
print(tab.round(5).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for (name, s), ls, col, mk in zip(scales.items(), LS, GREYS, MK):
    sub = tab[tab.level == name]
    ax[0].plot(sub.Delta, sub.auc_loss, ls=ls, color=col, marker=mk,
               label=f"{name} (AUC={sub.auc0.iloc[0]:.3f})")
    ax[1].plot(sub.Delta, sub.reversal, ls=ls, color=col, marker=mk, label=name)
ax[0].set_xlabel(r"lattice spacing $\Delta$"); ax[0].set_ylabel(r"AUC loss ($-\Delta$AUC)")
ax[0].legend(fontsize=7)
ax[1].set_xlabel(r"lattice spacing $\Delta$"); ax[1].set_ylabel("rank-reversal rate")
ax[1].legend(fontsize=7)
fig.tight_layout(); _save(fig, "02_ranking_stability"); plt.close(fig)

   level    auc0  Delta  auc_loss  reversal
baseline 0.86524   0.01  -0.00002   0.00073
baseline 0.86524   0.02   0.00015   0.00154
baseline 0.86524   0.05   0.00014   0.00417
baseline 0.86524   0.10   0.00108   0.00868
baseline 0.86524   0.20   0.00292   0.01964
baseline 0.86524   0.40   0.02256   0.06593
stronger 0.86524   0.01   0.00001   0.00059
stronger 0.86524   0.02   0.00003   0.00111
stronger 0.86524   0.05   0.00008   0.00243
stronger 0.86524   0.10   0.00047   0.00544
stronger 0.86524   0.20   0.00100   0.01060
stronger 0.86524   0.40   0.00446   0.02670
